In [1]:
#Inference time analysis - PyCaret
import time
import pandas as pd

from pycaret.classification import *
from sklearn.datasets import load_breast_cancer

#Load dataset
data = load_breast_cancer(as_frame=True)

df = data.frame

#Setup PyCaret
clf = setup(
    data=df,
    target="target",
    session_id=42,
    verbose=False
)

#Train best model
best_model = compare_models()

# Features only
X = df.drop("target", axis=1)

# Measure inference time
start = time.time()

predictions = predict_model(best_model, data=X)

end = time.time()

total_time = end - start

print("PyCaret best model:")
print(best_model)

print("\nTotal inference time:",
      round(total_time, 6),
      "seconds")

print("Average inference time per sample:",
      round(total_time / len(X), 8),
      "seconds")

,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC,TT (Sec)
rf,Random Forest Classifier,0.9725,0.9859,0.9800,0.9767,0.9781,0.9411,0.9419,0.0560
et,Extra Trees Classifier,0.9700,0.9925,0.9760,0.9764,0.9759,0.9361,0.9372,0.0500
lightgbm,Light Gradient Boosting Machine,0.9649,0.9885,0.9800,0.9653,0.9722,0.9244,0.9260,0.1320
ada,Ada Boost Classifier,0.9599,0.9832,0.9720,0.9650,0.9680,0.9144,0.9162,0.0440
xgboost,Extreme Gradient Boosting,0.9599,0.9877,0.9720,0.9645,0.9678,0.9147,0.9162,0.0320
gbc,Gradient Boosting Classifier,0.9575,0.9875,0.9680,0.9646,0.9657,0.9098,0.9117,0.1110
lda,Linear Discriminant Analysis,0.9573,0.9896,0.9960,0.9414,0.9675,0.9056,0.9106,0.0110
qda,Quadratic Discriminant Analysis,0.9549,0.9893,0.9600,0.9683,0.9635,0.9046,0.9067,0.0100
lr,Logistic Regression,0.9524,0.9922,0.9680,0.9578,0.9622,0.8977,0.9002,1.0690
ridge,Ridge Classifier,0.9524,0.9935,0.9920,0.9366,0.9633,0.8958,0.8996,0.0080


PyCaret best model:
RandomForestClassifier(bootstrap=True, ccp_alpha=0.0, class_weight=None,
                       criterion='gini', max_depth=None, max_features='sqrt',
                       max_leaf_nodes=None, max_samples=None,
                       min_impurity_decrease=0.0, min_samples_leaf=1,
                       min_samples_split=2, min_weight_fraction_leaf=0.0,
                       monotonic_cst=None, n_estimators=100, n_jobs=-1,
                       oob_score=False, random_state=42, verbose=0,
                       warm_start=False)

Total inference time: 0.179173 seconds
Average inference time per sample: 0.00031489 seconds


In [1]:
#TPOT inference time analysis
import time

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

from tpot import TPOTClassifier

#Load dataset
data = load_breast_cancer()

X = data.data
y = data.target

#Split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.3,
    random_state=42
)

#TPOT model
tpot = TPOTClassifier(
    generations=3,
    population_size=10,
    verbosity=0,
    random_state=42,
    cv=3,
    config_dict='TPOT light',
    n_jobs=1,
    disable_update_check=True
)

#Train
tpot.fit(X_train, y_train)

#Measure inference time
start = time.time()

predictions = tpot.predict(X_test)

end = time.time()

total_time = end - start

print("TPOT pipeline:")
print(tpot.fitted_pipeline_)

print("\nTotal inference time:",
      round(total_time, 6),
      "seconds")

print("Average inference time per sample:",
      round(total_time / len(X_test), 8),
      "seconds")

c:\Users\souha\Downloads\Human Centered AutoML Thesis\tpot_env\Lib\site-packages\tpot\builtins\__init__.py:36: UserWarning: Warning: optional dependency `torch` is not available. - skipping import of NN models.
  warnings.warn("Warning: optional dependency `torch` is not available. - skipping import of NN models.")


TPOT pipeline:
Pipeline(steps=[('stackingestimator',
                 StackingEstimator(estimator=DecisionTreeClassifier(max_depth=8,
                                                                    min_samples_leaf=13,
                                                                    min_samples_split=10,
                                                                    random_state=42))),
                ('logisticregression',
                 LogisticRegression(C=15.0, random_state=42))])

Total inference time: 0.001025 seconds
Average inference time per sample: 5.99e-06 seconds


In [1]:
# H2O inference time analysis

import time
import h2o

from h2o.automl import H2OAutoML
from sklearn.datasets import load_breast_cancer

# Start H2O
h2o.init()

# Load dataset
data = load_breast_cancer(as_frame=True)

df = data.frame

# Classification target
df["target"] = df["target"].astype("category")

# Convert to H2OFrame
hf = h2o.H2OFrame(df)

hf["target"] = hf["target"].asfactor()

# Split
train, test = hf.split_frame(ratios=[0.7], seed=42)

# Features and target
x = [col for col in hf.columns if col != "target"]
y = "target"

# Train AutoML
aml = H2OAutoML(
    max_models=10,
    seed=42,
    verbosity="info"
)

aml.train(
    x=x,
    y=y,
    training_frame=train
)

# Leader model
leader = aml.leader

# Measure inference time
start = time.time()

preds = leader.predict(test)

end = time.time()

total_time = end - start

print("H2O best model:")
print(leader.model_id)

print("\nTotal inference time:",
      round(total_time, 6),
      "seconds")

print("Average inference time per sample:",
      round(total_time / test.nrows, 8),
      "seconds")

Checking whether there is an H2O instance running at http://localhost:54321. connected.


H2O_cluster_uptime:,1 hour 20 mins
H2O_cluster_timezone:,Europe/Brussels
H2O_data_parsing_timezone:,UTC
H2O_cluster_version:,3.46.0.10
H2O_cluster_version_age:,2 months and 5 days
H2O_cluster_name:,H2O_from_python_souha_4mvstw
H2O_cluster_total_nodes:,1
H2O_cluster_free_memory:,3.835 Gb
H2O_cluster_total_cores:,12
H2O_cluster_allowed_cores:,12
H2O_cluster_status:,"locked, healthy"


Parse progress: |████████████████████████████████████████████████████████████████| (done) 100%
AutoML progress: |█
01:23:23.109: Project: AutoML_5_20260518_12323
01:23:23.110: 5-fold cross-validation will be used.
01:23:23.110: Setting stopping tolerance adaptively based on the training frame: 0.05
01:23:23.110: Build control seed: 42
01:23:23.111: training frame: Frame key: AutoML_5_20260518_12323_training_py_3_sid_aec1    cols: 31    rows: 391  chunks: 1    size: 33689  checksum: 8300906661992651134
01:23:23.111: validation frame: NULL
01:23:23.111: leaderboard frame: NULL
01:23:23.111: blending frame: NULL
01:23:23.111: response column: target
01:23:23.111: fold column: null
01:23:23.111: weights column: null
01:23:23.114: AutoML: XGBoost is not available; skipping it.
01:23:23.114: Loading execution steps: [{XGBoost : [def_2 (1g, 10w), def_1 (2g, 10w), def_3 (3g, 10w), grid_1 (4g, 90w), lr_search (7g, 30w)]}, {GLM : [def_1 (1g, 10w)]}, {DRF : [def_1 (2g, 10w), XRT (3g, 10w)]}, {GBM

Constrained AutoML Experiments


In [1]:
# Constrained TPOT experiment
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from tpot import TPOTClassifier
# Load dataset
data = load_breast_cancer()
X = data.data
y = data.target
# Split dataset
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.3,
    random_state=42
)

# Constrained TPOT
tpot = TPOTClassifier(
    generations=10,
    population_size=10,
    max_time_mins=10,
    cv=3,
    scoring="accuracy",
    random_state=42,
    verbosity=2,
    n_jobs=1,
    config_dict="TPOT light",
    disable_update_check=True
)

# Train
tpot.fit(X_train, y_train)

# Predict
predictions = tpot.predict(X_test)

# Accuracy
accuracy = accuracy_score(y_test, predictions)

print("Constrained TPOT pipeline:")
print(tpot.fitted_pipeline_)

print("\nAccuracy:", round(accuracy, 4))

c:\Users\souha\Downloads\Human Centered AutoML Thesis\tpot_env\Lib\site-packages\tpot\builtins\__init__.py:36: UserWarning: Warning: optional dependency `torch` is not available. - skipping import of NN models.
  warnings.warn("Warning: optional dependency `torch` is not available. - skipping import of NN models.")


                                                                            
Generation 1 - Current best internal CV score: 0.9446722867775499
                                                                            
Generation 2 - Current best internal CV score: 0.9698108908635225
                                                                            
Generation 3 - Current best internal CV score: 0.9698108908635225
                                                                            
Generation 4 - Current best internal CV score: 0.9698108908635225
                                                                            
Generation 5 - Current best internal CV score: 0.9698108908635225
                                                                            
Generation 6 - Current best internal CV score: 0.9698108908635225
                                                                            
Generation 7 - Current best internal CV score: 0.9748424090529353